# Laboratory: Anomaly Detection on MVTec AD Dataset
## Reconstruction-based, student-teacher and feature-based methods

This notebook is a **Google Colab laboratory** about industrial anomaly detection for the Advanced Deep Learning Course @ Politecnico di Milano.

We use the **MVTec AD** benchmark and implement three complementary families of methods:

1. **Autoencoders** as **reconstruction-based** detectors  
   - one convolutional latent-space autoencoder trained with **pixel-wise MSE**
   - one bottleneck autoencoder trained with **pixel-wise MSE**

2. **Uninformed Students** as a **student-teacher** approach  
   - a teacher produces local patch descriptors
   - an ensemble of students learns to regress the teacher on **normal images only**
   - anomaly scores combine **regression error** and **predictive variance**

3. **PatchCore Lite** as a **feature-based** detector  
   - nearest-neighbor anomaly detection in a memory bank of normal patch features
   - based on a pretrained **WideResNet50 / ResNet18**

4. A **Self Supervised Method** for Anomaly Detection  
   - use a pretext task for learning a representation of a single class (one-class classification)

---

## Learning goals

By the end of the lab, you should understand:

- why MVTec AD is commonly used for anomaly detection
- the most common metrics used in anomaly localization and anomaly detection
- how **one-class training** works (train only on *normal* images)
- why purely reconstruction-based methods can struggle on subtle industrial defects
- how **student-teacher regression** can turn descriptor prediction errors into anomaly maps
- why feature-based methods such as PatchCore often work very well on MVTec-like data
- how anomaly scores can arise from **reconstruction error**, **student disagreement / variance**, or **distances in a memory bank of normal patch features**

---

## Main references

1. Bergmann, P. et al.  
   **The MVTec Anomaly Detection Dataset: A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection**  
   IJCV 2021  
   https://link.springer.com/content/pdf/10.1007/s11263-020-01400-4.pdf

2. Bergmann, P., Fauser, M., Sattlegger, D., and Steger, C.  
   **Uninformed Students: Student–Teacher Anomaly Detection with Discriminative Latent Embeddings**  
   CVPR 2020  
   https://openaccess.thecvf.com/content_CVPR_2020/papers/Bergmann_Uninformed_Students_Student-Teacher_Anomaly_Detection_With_Discriminative_Latent_Embeddings_CVPR_2020_paper.pdf

3. Roth, K. et al.  
   **Towards Total Recall in Industrial Anomaly Detection**  
   CVPR 2022   
   Reference implementation: https://github.com/amazon-science/patchcore-inspection

4. I. Golan, and R. El-Yaniv.   
   **Deep Anomaly Detection Using Geometric Transformations.**   
   NeurIPS 2018
   https://proceedings.neurips.cc/paper_files/paper/2018/file/5e62d03aec0d17facfc5355dd90d441c-Paper.pdf

---

## Practical note

This notebook is intentionally written in a **single-class / one-category** setting: choose one MVTec class (for example `capsule`, `bottle`, `wood`, or `screw`) and train **one model for that class only**.

Run the notebook from top to bottom. A **GPU runtime** is strongly recommended for all the sections if you want to train the models.

In [ ]:
# Import Cell

import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name.split(">=")[0].split("==")[0].replace("-", "_")
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

ensure_package("timm>=1.0.0", "timm")
ensure_package("scikit-learn>=1.1.0", "sklearn")

import copy
import math
import os
import random
import subprocess
import tarfile
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from tqdm.auto import tqdm
import scipy.ndimage as ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchsummary import summary

import timm
from pytorch_msssim import ms_ssim
from google.colab import drive

import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
from PIL import Image, ImageEnhance, ImageOps

In [ ]:
# Fix random seeds for reproducibility and checking runtime resources

SEED = 7

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 2 if os.name != "nt" else 0

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# We mount the user drive folder, and we assume the dataset .tar file is in user personal drive
# If not it will try to download it from official dataset page, either in the full way, or just a class of the dataset
PERSONAL_DRIVE_FOLDER = '/content/drive/'
drive.mount(PERSONAL_DRIVE_FOLDER)

# Official MVTec AD download links (from the official dataset page).
FULL_DATASET_URL = (
    "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/"
    "download/420938113-1629960298/mvtec_anomaly_detection.tar.xz"
)

CLASS_URLS = {
    "bottle": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937370-1629958698/bottle.tar.xz",
    "cable": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937413-1629958794/cable.tar.xz",
    "capsule": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937454-1629958872/capsule.tar.xz",
    "carpet": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937484-1629959013/carpet.tar.xz",
    "grid": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937487-1629959044/grid.tar.xz",
    "hazelnut": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937545-1629959162/hazelnut.tar.xz",
    "leather": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937607-1629959262/leather.tar.xz",
    "metal_nut": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937637-1629959294/metal_nut.tar.xz",
    "pill": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938129-1629960351/pill.tar.xz",
    "screw": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938130-1629960389/screw.tar.xz",
    "tile": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938133-1629960456/tile.tar.xz",
    "toothbrush": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938134-1629960477/toothbrush.tar.xz",
    "transistor": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938166-1629960554/transistor.tar.xz",
    "wood": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938383-1629960649/wood.tar.xz",
    "zipper": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420938385-1629960680/zipper.tar.xz",
}

In [ ]:
# Utilities to load MVTec AD dataset

def list_image_files(folder):
    folder = Path(folder)
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts])

def safe_extract_tar_xz(archive_path, destination, class_name=None):
    archive_path = Path(archive_path)
    destination = Path(destination).resolve()

    # Shallow check on the class
    if class_name:
      if os.path.exists(destination / Path(class_name)):
        print(f"Dataset already extracted for {class_name}!")
      else:
        with tarfile.open(archive_path, "r:xz") as tar:
            for member in tar.getmembers():
                member_path = (destination / member.name).resolve()
                if not str(member_path).startswith(str(destination)):
                    raise RuntimeError("Unsafe path detected while extracting the tar archive.")
            tar.extractall(destination)
    else:
      # Shallow check on the full folder
      if os.path.exists(destination):
        print("Dataset already extracted!")
      else:
        with tarfile.open(archive_path, "r:xz") as tar:
            for member in tar.getmembers():
                member_path = (destination / member.name).resolve()
                if not str(member_path).startswith(str(destination)):
                    raise RuntimeError("Unsafe path detected while extracting the tar archive.")
            tar.extractall(destination)

def download_file(url, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        print(f"Archive already exists: {output_path}")
        return
    print(f"Downloading to {output_path} ...")
    subprocess.run(["wget", "-c", url, "-O", str(output_path)], check=True)

def maybe_download_mvtec(root, mode="skip", class_name="bottle"):
    root = Path(root)

    if mode == "skip":
      if os.path.exists(root / "mvtec_anomaly_detection"):
        print("Dataset already present!")
        return root / Path("mvtec_anomaly_detection")
      if (root / "mvtec_anomaly_detection.tar.xz").is_file():
        safe_extract_tar_xz(root / "mvtec_anomaly_detection.tar.xz", root / "mvtec_anomaly_detection")
        print(f"Found and extracted data at: {root / Path("mvtec_anomaly_detection")}")
        return root / Path("mvtec_anomaly_detection")
      else:
        raise FileNotFoundError(f"DOWNLOAD_MODE='skip', but dataset was not found under {root}.")


    if mode == "single":
        if class_name not in CLASS_URLS:
            raise KeyError(f"Unknown MVTec class: {class_name}")
        url = CLASS_URLS[class_name]
        archive_path = root / f"{class_name}.tar.xz"
        download_file(url, archive_path)
        print("Extracting archive ...")
        safe_extract_tar_xz(archive_path, root / Path("mvtec_anomaly_detection"), class_name)
    elif mode == "full":
        url = FULL_DATASET_URL
        archive_path = root / "mvtec_anomaly_detection.tar.xz"
        download_file(url, archive_path)
        print("Extracting archive ...")
        safe_extract_tar_xz(archive_path, root / Path("mvtec_anomaly_detection"))
    else:
        raise ValueError("mode must be one of: 'single', 'full', 'skip'")


    print(f"Dataset ready at: {root / Path("mvtec_anomaly_detection")}")
    return root / Path("mvtec_anomaly_detection")

def summarize_mvtec_class(root, class_name):
    class_root = Path(root) / class_name
    rows = []

    train_good = list_image_files(class_root / "train" / "good")
    rows.append({"split": "train", "defect_type": "good", "n_images": len(train_good)})

    for defect_dir in sorted((class_root / "test").iterdir()):
        if defect_dir.is_dir():
            rows.append(
                {
                    "split": "test",
                    "defect_type": defect_dir.name,
                    "n_images": len(list_image_files(defect_dir)),
                }
            )
    return pd.DataFrame(rows)

In [ ]:
# We print some stats about a specific class to check everything is fine

# Which MVTec category to study.
CLASS_NAME = "capsule"   # try also: "capsule", "hazelnut", "screw", "wood", ...

# Download options:
# - "single": download only the selected class archive
# - "full"  : download the whole MVTec AD dataset archive
# - "skip"  : assume data is already present under DATA_ROOT
DOWNLOAD_MODE = "single"

# Where the dataset will live in Colab.
MVTEC_ROOT = maybe_download_mvtec('/content/drive/MyDrive/', DOWNLOAD_MODE, class_name=CLASS_NAME)

summary_df = summarize_mvtec_class(MVTEC_ROOT, CLASS_NAME)
print(f"MVTec Class {CLASS_NAME}")
display(summary_df)

In [ ]:
# =========================
# User configuration
# =========================

# Image resolution used in this laboratory
IMAGE_SIZE = 256

# Split normal training images into train/validation for threshold calibration.
VAL_RATIO = 0.15

# Visualization
N_VIS_SAMPLES = 5

# Datasets batch sizes
AE_BATCH_SIZE = 32
PATCHCORE_BATCH_SIZE = 16

We create the dataset class to load the **MVTec AD** classes and train the models.

In [ ]:
def pil_to_tensor(img):
    arr = np.asarray(img, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[..., None]
    arr = arr / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)

class MVTecSingleClassDataset(Dataset):
    def __init__(self, root, class_name, split="train", image_size=256):
        self.root = Path(root)
        self.class_name = class_name
        self.split = split
        self.image_size = image_size
        self.samples = []
        self.class_root = self.root / class_name

        if split not in {"train", "test"}:
            raise ValueError("split must be 'train' or 'test'")

        if split == "train":
            good_dir = self.class_root / "train" / "good"
            for img_path in list_image_files(good_dir):
                self.samples.append(
                    {
                        "image_path": img_path,
                        "mask_path": None,
                        "label": 0,
                        "defect_type": "good",
                    }
                )
        else:
            test_root = self.class_root / "test"
            gt_root = self.class_root / "ground_truth"
            for defect_dir in sorted(test_root.iterdir()):
                if not defect_dir.is_dir():
                    continue
                defect_type = defect_dir.name
                for img_path in list_image_files(defect_dir):
                    if defect_type == "good":
                        mask_path = None
                        label = 0
                    else:
                        mask_name = f"{img_path.stem}_mask.png"
                        mask_path = gt_root / defect_type / mask_name
                        label = 1
                    self.samples.append(
                        {
                            "image_path": img_path,
                            "mask_path": mask_path,
                            "label": label,
                            "defect_type": defect_type,
                        }
                    )

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No samples found for class='{class_name}', split='{split}' under {self.class_root}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        record = self.samples[idx]

        image = Image.open(record["image_path"]).convert("RGB")
        image = image.resize((self.image_size, self.image_size), resample=Image.BILINEAR)
        image = pil_to_tensor(image)

        if record["mask_path"] is None:
            mask = torch.zeros((1, self.image_size, self.image_size), dtype=torch.float32)
        else:
            mask = Image.open(record["mask_path"]).convert("L")
            mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)
            mask = (pil_to_tensor(mask) > 0.5).float()

        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(record["label"], dtype=torch.long),
            "defect_type": record["defect_type"],
            "path": str(record["image_path"]),
        }

def split_train_val(dataset, val_ratio=0.15, seed=SEED):
    n = len(dataset)
    indices = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    n_val = max(1, int(round(n * val_ratio)))
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]
    return Subset(dataset, train_idx.tolist()), Subset(dataset, val_idx.tolist())

train_full_dataset = MVTecSingleClassDataset(MVTEC_ROOT, CLASS_NAME, split="train", image_size=IMAGE_SIZE)
test_dataset = MVTecSingleClassDataset(MVTEC_ROOT, CLASS_NAME, split="test", image_size=IMAGE_SIZE)
train_dataset, val_dataset = split_train_val(train_full_dataset, val_ratio=VAL_RATIO, seed=SEED)

print(f"Train normals (full): {len(train_full_dataset)}")
print(f"Train normals (used for fitting): {len(train_dataset)}")
print(f"Validation normals: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=AE_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=AE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=PATCHCORE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

patchcore_train_loader = DataLoader(
    train_dataset,
    batch_size=PATCHCORE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)


In the following cell we have defined some utilities to plot the results

In [ ]:
def tensor_to_numpy_image(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(x, 0.0, 1.0)

def tensor_to_numpy_mask(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().squeeze().numpy()
    return x.astype(np.float32)

def normalize_map(amap):
    amap = np.asarray(amap, dtype=np.float32)
    if amap.max() <= amap.min():
        return np.zeros_like(amap)
    return (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

def heatmap_overlay(image, amap, alpha=0.45):
    image = tensor_to_numpy_image(image)
    heat = plt.cm.jet(normalize_map(amap))[..., :3]
    return np.clip((1.0 - alpha) * image + alpha * heat, 0.0, 1.0)

def show_dataset_examples(train_dataset, test_dataset, n_train=4, n_test=4):
    train_indices = np.linspace(0, len(train_dataset) - 1, n_train, dtype=int)
    test_indices = np.linspace(0, len(test_dataset) - 1, n_test, dtype=int)

    fig, axes = plt.subplots(2, max(n_train, n_test), figsize=(4 * max(n_train, n_test), 7))
    axes = np.atleast_2d(axes)

    for col, idx in enumerate(train_indices):
        sample = train_dataset[idx]
        axes[0, col].imshow(tensor_to_numpy_image(sample["image"]))
        axes[0, col].set_title(f"Train normal\n{Path(sample['path']).name}")
        axes[0, col].axis("off")

    for col, idx in enumerate(test_indices):
        sample = test_dataset[idx]
        img = tensor_to_numpy_image(sample["image"])
        mask = tensor_to_numpy_mask(sample["mask"])
        axes[1, col].imshow(img)
        if sample["label"].item() == 1:
            axes[1, col].contour(mask, levels=[0.5], colors="red", linewidths=1.5)
        axes[1, col].set_title(f"Test: {sample['defect_type']}\nlabel={sample['label'].item()}")
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()

show_dataset_examples(train_full_dataset, test_dataset)


Next we define some utilities to compute the metrics (Image-Level AUROC, Pixel-Level AUROC, Image-Level Average Precision, Pixel-Level Average Precision, and best thresholds computed as Youden statistic using Image-Level and Pixel-Level AUROC)

In [ ]:
def safe_metric(metric_fn, y_true, scores):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(metric_fn(y_true, scores))

def youden_threshold(y_true, y_score):
    """
    Return the threshold that maximizes the Youden statistic:
        J = sensitivity + specificity - 1 = TPR - FPR

    Returns np.nan when the threshold cannot be computed
    (for example, if only one class is present).
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # ROC/Youden is undefined if only one class is present
    if y_true.size == 0 or np.unique(y_true).size < 2:
        return np.nan

    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    return float(thresholds[best_idx])

def summarize_metrics(results):
    image_labels = np.array([r["label"] for r in results], dtype=np.int64)
    image_scores = np.array([r["score"] for r in results], dtype=np.float32)

    pixel_masks = np.concatenate([r["mask"].reshape(-1) for r in results]).astype(np.uint8)
    pixel_scores = np.concatenate([r["anomaly_map"].reshape(-1) for r in results]).astype(np.float32)

    metrics = {
        "image_auroc": safe_metric(roc_auc_score, image_labels, image_scores),
        "image_ap": safe_metric(average_precision_score, image_labels, image_scores),
        "image_threshold": youden_threshold(image_labels, image_scores),
        "pixel_auroc": safe_metric(roc_auc_score, pixel_masks, pixel_scores),
        "pixel_ap": safe_metric(average_precision_score, pixel_masks, pixel_scores),
        "pixel_threshold": youden_threshold(pixel_masks, pixel_scores),
    }
    return metrics

def summarize_metrics_image_only(results):
  image_labels = np.array([r["label"] for r in results], dtype=np.int64)
  image_scores = np.array([r["score"] for r in results], dtype=np.float32)

  metrics = {
      "image_auroc": safe_metric(roc_auc_score, image_labels, image_scores),
      "image_ap": safe_metric(average_precision_score, image_labels, image_scores),
      "image_threshold": youden_threshold(image_labels, image_scores),
  }
  return metrics

def plot_score_distribution(results, title="Image-level anomaly scores"):
    normal_scores = [r["score"] for r in results if r["label"] == 0]
    anomaly_scores = [r["score"] for r in results if r["label"] == 1]

    plt.figure(figsize=(7, 4))
    plt.hist(normal_scores, bins=20, alpha=0.7, label="normal")
    plt.hist(anomaly_scores, bins=20, alpha=0.7, label="anomalous")
    plt.xlabel("image anomaly score")
    plt.ylabel("count")
    plt.title(title)
    plt.legend()
    plt.show()

def plot_score_distribution_v2(results, title="Image-level anomaly scores"):
    normal_scores = [r["score"] for r in results if r["label"] == 0]
    anomaly_scores = [r["score"] for r in results if r["label"] == 1]

    bins = np.linspace(
        min(normal_scores + anomaly_scores),
        max(normal_scores + anomaly_scores),
        20
    )

    plt.figure(figsize=(7, 4))
    plt.hist(normal_scores, bins=bins, density=True, alpha=0.5, label="normal")
    plt.hist(anomaly_scores, bins=bins, density=True, alpha=0.5, label="anomalous")

    plt.xlabel("image anomaly score")
    plt.ylabel("density")
    plt.title(title)
    plt.legend()
    plt.show()

def show_detection_results(results, n=5, include_reconstruction=False, thresholds=None, title=None):
    if len(results) == 0:
        return

    # You can change the following line to visualize the worst n examples or the fixed ones
    #order = np.argsort([r["score"] for r in results])[::-1]
    #chosen = order[: min(n, len(order))]

    # These examples are thought for capsule class
    chosen = [0, 23, 68, 89, 112]

    n_cols = 5 if include_reconstruction else 4
    fig, axes = plt.subplots(len(chosen), n_cols, figsize=(4 * n_cols, 3.7 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    if title is not None:
        fig.suptitle(title, fontsize=16, y=1.02)

    for row, idx in enumerate(chosen):
        r = results[idx]
        img = r["image"]
        amap = r["anomaly_map"]
        mask = r["mask"]

        col = 0
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"Input\nlabel={r['label']} score={r['score']:.4f}")
        axes[row, col].axis("off")
        col += 1

        if include_reconstruction:
            axes[row, col].imshow(r["reconstruction"])
            axes[row, col].set_title("Reconstruction")
            axes[row, col].axis("off")
            col += 1

        axes[row, col].imshow(heatmap_overlay(img, amap))
        axes[row, col].set_title("Overlay")
        axes[row, col].axis("off")
        col += 1

        im = axes[row, col].imshow(amap, cmap="jet")
        axes[row, col].set_title("Anomaly map")
        axes[row, col].axis("off")
        plt.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)
        col += 1

        if thresholds is None:
            axes[row, col].imshow(mask, cmap="gray")
            axes[row, col].set_title("GT mask")
            axes[row, col].axis("off")
        else:
            pred_mask = (amap >= thresholds["pixel_threshold"]).astype(np.float32)
            panel = np.concatenate([mask, pred_mask], axis=1)
            axes[row, col].imshow(panel, cmap="gray")
            axes[row, col].set_title("GT mask | Pred mask")
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

def _gaussian_kernel(window_size=11, sigma=1.5, channels=1, device="cpu", dtype=torch.float32):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel_2d = torch.outer(g, g)
    return kernel_2d.expand(channels, 1, window_size, window_size).contiguous()

def _filter2d(x, kernel):
    pad = kernel.shape[-1] // 2
    x = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    return F.conv2d(x, kernel, groups=x.shape[1])

def gaussian_blur(x, sigma=4.0):
    if sigma is None or sigma <= 0:
        return x
    kernel_size = int(max(3, 2 * round(4 * sigma) + 1))
    kernel_size += 1 - (kernel_size % 2)  # keep it odd
    kernel = _gaussian_kernel(
        window_size=kernel_size,
        sigma=float(sigma),
        channels=x.shape[1],
        device=x.device,
        dtype=x.dtype,
    )
    return _filter2d(x, kernel)



## Part 1 — Autoencoders as reconstruction-based anomaly detectors

The core idea is simple:

- train an autoencoder **only on normal images**
- ask it to reconstruct a test image
- if the image is normal, reconstruction should be good
- if the image contains an anomaly, reconstruction should be worse
- use the reconstruction discrepancy as an anomaly map

We train two models:

1. **MSE / L2 autoencoder with Bottleneck**  
   
2. **MSE / L2 autoencoder with Convolutional Latent Space**  

### Anomaly Map
anomaly map = per-pixel squared error averaged across channels

### Important limitation

A strong autoencoder can sometimes reconstruct anomalous content *too well*.  
When that happens, the anomaly score becomes small and detection gets harder. This is one reason why purely reconstruction-based methods are often weaker than strong feature-based methods on MVTec.


In [ ]:
# Autoencoders settings

AE_LR = 0.001
AE_BATCH_SIZE = 32
AE_EPOCHS = 20

# Gaussian smoothing applied to anomaly maps.
ANOMALY_MAP_SIGMA = 4.0

In [ ]:
class Encoder(nn.Module):
    """Shared encoder architecture for all tasks"""
    def __init__(self, latent_dim=128):
        super(Encoder, self).__init__()

       # Convolutional feature extractor
        ...

        if latent_dim != 0:

          ...

        self.latent_dim = latent_dim

    def forward(self, x):
        x = self.features(x)
        if self.latent_dim != 0:
          x = self.fc(x)
        return x


In [ ]:
class ConvAutoencoder(nn.Module):
    """Convolutional Autoencoder using shared Encoder"""
    def __init__(self, encoder):
        super(ConvAutoencoder, self).__init__()
        self.encoder = encoder
        self.latent_dim = encoder.latent_dim

        if self.latent_dim != 0:
            # Latent → 512 x H_l / 32 x W_l / 32
            ...

        # Decoder: symmetric to encoder
        ...

    def forward(self, x):
        latent = self.encoder(x)
        if self.latent_dim != 0:
          latent = self.fc(latent)
        reconstruction = self.decoder(latent)
        return reconstruction


Now we define the training and evaluation utilities

In [ ]:
def plot_history(history_df, title="Training history"):
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.show()

def train_autoencoder(model, epochs=AE_EPOCHS, lr=AE_LR, use_improved_recon_loss=False, lambda_l1=0.1, lambda_ssim=0.9):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for batch in tqdm(train_loader, desc=f"AE train epoch {epoch}/{epochs}", leave=False):
            ...

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                ...

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"[AE] epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df

@torch.no_grad()
def predict_autoencoder(model, loader):
    model.eval()
    results = []

    for batch in tqdm(loader, desc=f"Autoencoder inference"):
        x = batch["image"].to(DEVICE, non_blocking=True)
        recon = model(x)

        anomaly_map = (x - recon).pow(2).mean(dim=1, keepdim=True)

        anomaly_map = gaussian_blur(anomaly_map, sigma=ANOMALY_MAP_SIGMA)
        image_scores = anomaly_map.flatten(1).amax(dim=1)

        for i in range(x.shape[0]):
            results.append(
                {
                    "image": tensor_to_numpy_image(x[i]),
                    "reconstruction": tensor_to_numpy_image(recon[i]),
                    "anomaly_map": anomaly_map[i, 0].detach().cpu().numpy().astype(np.float32),
                    "mask": tensor_to_numpy_mask(batch["mask"][i]),
                    "label": int(batch["label"][i].item()),
                    "score": float(image_scores[i].item()),
                    "defect_type": batch["defect_type"][i],
                    "path": batch["path"][i],
                }
            )
    return results


In [ ]:
# Create the Bottleneck Autoencoder
bott_autoencoder = ConvAutoencoder(Encoder(latent_dim=256).to(DEVICE)).to(DEVICE)
print(f"Convolutional Latent Space Autoencoder parameters: {sum(p.numel() for p in bott_autoencoder.parameters()):,}")
summary(bott_autoencoder, input_size=(3, 256, 256))

In [ ]:
# Create the Convolutional Latent Space Autoencoder
conv_autoencoder = ConvAutoencoder(Encoder(latent_dim=0).to(DEVICE)).to(DEVICE)
print(f"Convolutional Latent Space Autoencoder parameters: {sum(p.numel() for p in conv_autoencoder.parameters()):,}")
summary(conv_autoencoder, input_size=(3, 256, 256))

In [ ]:
# Uncomment the following lines if you want to train
torch.cuda.empty_cache()

#bott_autoencoder_mse, bott_autoencoder_mse_history = train_autoencoder(bott_autoencoder, epochs=40, lr=AE_LR)
#plot_history(bott_autoencoder_mse_history, title="Bottleneck Autoencoder (MSE) training history")
#torch.save(bott_autoencoder_mse, "/content/drive/MyDrive/bott_autoencoder_40epochs_256.pth")
#with open("/content/drive/MyDrive/bott_autoencoder_40epochs_256_history.pkl", "wb") as f:
#    pickle.dump(bott_autoencoder_mse_history, f)

# Comment if you want to train from scratch
bott_autoencoder_mse = torch.load(
    "/content/drive/MyDrive/bott_autoencoder_40epochs_256.pth", weights_only=False, map_location=torch.device(DEVICE)
).to(DEVICE)
with open("/content/drive/MyDrive/bott_autoencoder_40epochs_256_history.pkl", "rb") as f:
    bott_autoencoder_mse_history = pickle.load(f)
plot_history(bott_autoencoder_mse_history, title="Bottleneck Autoencoder (MSE) training history")

# Evaluate
bott_autoencoder_mse_test_results = predict_autoencoder(bott_autoencoder_mse, test_loader)
bott_autoencoder_mse_metrics = summarize_metrics(bott_autoencoder_mse_test_results)
bott_autoencoder_mse_thresholds = {k: bott_autoencoder_mse_metrics[k] for k in ("pixel_threshold", "image_threshold")}
print("Bottleneck Autoencoder (MSE) metrics:")
display(pd.DataFrame([bott_autoencoder_mse_metrics]).round(4))

plot_score_distribution_v2(bott_autoencoder_mse_test_results, title="MSE bott autoencoder image scores")
show_detection_results(
    bott_autoencoder_mse_test_results,
    n=N_VIS_SAMPLES,
    include_reconstruction=True,
    thresholds=bott_autoencoder_mse_thresholds,
    title="Bottleneck Autoencoder (MSE): test examples",
)

In [ ]:
# Uncomment the following lines if you want to train
torch.cuda.empty_cache()

#conv_autoencoder_mse, conv_autoencoder_mse_history = train_autoencoder(conv_autoencoder, epochs=40, lr=AE_LR, use_improved_recon_loss=True)
#plot_history(conv_autoencoder_mse_history, title="Convolutional Autoencoder (MSE) training history")
#torch.save(conv_autoencoder_mse, "/content/drive/MyDrive/conv_autoencoder_40epochs_improved_recon_v3.pth")
#with open("/content/drive/MyDrive/conv_autoencoder_40epochs_improved_recon_v3_history.pkl", "wb") as f:
#    pickle.dump(conv_autoencoder_mse_history, f)

# Comment if you want to train from scratch
conv_autoencoder_mse = torch.load(
    "/content/drive/MyDrive/conv_autoencoder_40epochs_improved_recon_v3.pth", weights_only=False, map_location=torch.device(DEVICE)
).to(DEVICE)
with open("/content/drive/MyDrive/conv_autoencoder_40epochs_improved_recon_v3_history.pkl", "rb") as f:
    conv_autoencoder_mse_history = pickle.load(f)
plot_history(conv_autoencoder_mse_history, title="Convolutional Autoencoder (IRL) training history")

# Evaluate
conv_autoencoder_mse_test_results = predict_autoencoder(conv_autoencoder_mse, test_loader)
conv_autoencoder_mse_metrics = summarize_metrics(conv_autoencoder_mse_test_results)
conv_autoencoder_mse_thresholds = {k: conv_autoencoder_mse_metrics[k] for k in ("pixel_threshold", "image_threshold")}
print("Convolutional Autoencoder (IRL) metrics:")
display(pd.DataFrame([conv_autoencoder_mse_metrics]).round(4))

plot_score_distribution_v2(conv_autoencoder_mse_test_results, title="IRL conv autoencoder image scores")
show_detection_results(
    conv_autoencoder_mse_test_results,
    n=N_VIS_SAMPLES,
    include_reconstruction=True,
    thresholds=conv_autoencoder_mse_thresholds,
    title="Convolutional Autoencoder (IRL): test examples",
)

## Part 2 — Uninformed Students: student-teacher anomaly detection

We now add a student-teacher method inspired by **Bergmann et al., CVPR 2020**.

The idea is the following:

- train a **teacher** network that outputs a dense local descriptor for each image pixel
- train an ensemble of **students** **only on anomaly-free images** so that they regress the teacher descriptors
- during inference, anomalous regions tend to produce:
  1. **large regression error** with respect to the teacher
  2. **large predictive variance** across the students

### Practical choices for this notebook

- We stay in the **single-class / one-category** MVTec setting (`CLASS_NAME`), to train the students
- In the original paper, the teacher is pretrained on random **ImageNet** patches while students are trained on anomaly-free images from the target dataset. To keep this notebook more self-contained, we use as teacher **ImageNet-pretrained ResNet-18**.

### Final anomaly score

For each pixel, the final anomaly map combines:

- the **teacher-student regression error**
- the **student ensemble predictive variance**

Both scores are normalized on the anomaly-free validation set before they are summed.

In [ ]:
# -------------------------
# Uninformed Students settings
# -------------------------

US_PATCH_SIZES = (17,)          # set to (17, 33, 65) for the full multi-scale variant
US_N_STUDENTS = 3               # ensemble size M in the paper
US_TEACHER_EPOCHS = 30          # shortened schedule for notebook runtime
US_STUDENT_EPOCHS = 20          # shortened schedule for notebook runtime
US_TEACHER_LR = 2e-4
US_STUDENT_LR = 1e-4
US_WEIGHT_DECAY = 1e-5
US_TEACHER_IMAGE_BATCH_SIZE = 8 # 8 images x 8 patches -> 64 random patches / step
US_STUDENT_BATCH_SIZE = 1       # close to the original full-image training setup
US_PATCHES_PER_IMAGE = 16
US_AUGMENT_GRAY_PROB = 0.1
US_AUGMENT_NOISE_STD = 0.05
US_COMPACTNESS_WEIGHT = 1.0
US_ANOMALY_MAP_SIGMA = 0.0      # keep 0.0 for a score map closer to the paper

US_IMAGENET_MEAN = [0.485, 0.456, 0.406]
US_IMAGENET_STD = [0.229, 0.224, 0.225]


def us_make_loader(dataset, batch_size=1, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def us_count_parameters(model):
    return sum(p.numel() for p in model.parameters())

def rgb_to_gray(x):
    if x.shape[1] == 1:
        return x
    weights = x.new_tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)
    return (x * weights).sum(dim=1, keepdim=True)

def us_augment_patches(patches, gray_prob=US_AUGMENT_GRAY_PROB, noise_std=US_AUGMENT_NOISE_STD):
    patches = patches.clone()

    flip_h = torch.rand(patches.shape[0], device=patches.device) < 0.5
    if flip_h.any():
        patches[flip_h] = torch.flip(patches[flip_h], dims=[-1])

    flip_v = torch.rand(patches.shape[0], device=patches.device) < 0.5
    if flip_v.any():
        patches[flip_v] = torch.flip(patches[flip_v], dims=[-2])

    to_gray = torch.rand(patches.shape[0], device=patches.device) < gray_prob
    if to_gray.any():
        gray = rgb_to_gray(patches[to_gray]).repeat(1, 3, 1, 1)
        patches[to_gray] = gray

    if noise_std is not None and noise_std > 0:
        patches = (patches + noise_std * torch.randn_like(patches)).clamp(0.0, 1.0)

    return patches


def us_sample_random_patches(images, patch_size, patches_per_image=US_PATCHES_PER_IMAGE, augment=True):
    b, _, h, w = images.shape
    if h < patch_size or w < patch_size:
        raise ValueError(f"patch_size={patch_size} is larger than the image size {(h, w)}")

    patches = []
    max_top = h - patch_size + 1
    max_left = w - patch_size + 1

    for img_idx in range(b):
        for _ in range(patches_per_image):
            top = int(torch.randint(0, max_top, (1,), device=images.device).item())
            left = int(torch.randint(0, max_left, (1,), device=images.device).item())
            patches.append(images[img_idx:img_idx + 1, :, top:top + patch_size, left:left + patch_size])

    patches = torch.cat(patches, dim=0)
    if augment:
        patches = us_augment_patches(patches)
    return patches


def us_distillation_loss(pred, target):
    return ((pred - target) ** 2).sum(dim=1).mean()


def us_compactness_loss(desc, eps=1e-6):
    if desc.shape[0] < 2:
        return desc.new_tensor(0.0)
    z = desc - desc.mean(dim=0, keepdim=True)
    z = z / (desc.std(dim=0, unbiased=False, keepdim=True) + eps)
    corr = z.T @ z / max(1, z.shape[0] - 1)
    off_diag = corr - torch.diag(torch.diag(corr))
    return (off_diag ** 2).mean()


def us_student_loss(student_out, teacher_target):
    return ((student_out - teacher_target) ** 2).sum(dim=-1).mean()


def us_regression_error_map(students_pred, teacher_pred):
    mean_students = students_pred.mean(dim=1)
    return ((mean_students - teacher_pred) ** 2).sum(dim=-1)


def us_variance_map(students_pred):
    mean_sq_norm = (students_pred ** 2).sum(dim=-1).mean(dim=1)
    mean_students = students_pred.mean(dim=1)
    sq_norm_mean = (mean_students ** 2).sum(dim=-1)
    return mean_sq_norm - sq_norm_mean


def us_update_vector_stats(sum_vec, sum_sq_vec, count, x):
    x = x.reshape(-1, x.shape[-1]).float()
    batch_sum = x.sum(dim=0)
    batch_sum_sq = (x ** 2).sum(dim=0)
    if sum_vec is None:
        sum_vec = batch_sum
        sum_sq_vec = batch_sum_sq
    else:
        sum_vec = sum_vec + batch_sum
        sum_sq_vec = sum_sq_vec + batch_sum_sq
    count += x.shape[0]
    return sum_vec, sum_sq_vec, count


def us_finalize_vector_stats(sum_vec, sum_sq_vec, count, eps=1e-6):
    mean = sum_vec / max(count, 1)
    var = (sum_sq_vec / max(count, 1)) - mean.pow(2)
    var = var.clamp_min(eps)
    return mean, var, torch.sqrt(var)


def us_update_scalar_stats(sum_val, sum_sq_val, count, x):
    x = x.reshape(-1).float()
    batch_sum = x.sum()
    batch_sum_sq = (x ** 2).sum()
    if sum_val is None:
        sum_val = batch_sum
        sum_sq_val = batch_sum_sq
    else:
        sum_val = sum_val + batch_sum
        sum_sq_val = sum_sq_val + batch_sum_sq
    count += x.numel()
    return sum_val, sum_sq_val, count


def us_finalize_scalar_stats(sum_val, sum_sq_val, count, eps=1e-6):
    mean = sum_val / max(count, 1)
    var = (sum_sq_val / max(count, 1)) - mean.pow(2)
    var = var.clamp_min(eps)
    return mean, var, torch.sqrt(var)

In [ ]:
class MultiPoolPrepare(nn.Module):
    def __init__(self, patch_y, patch_x):
        super().__init__()
        pady = patch_y - 1
        padx = patch_x - 1
        self.pad_top = int(np.ceil(pady / 2))
        self.pad_bottom = int(np.floor(pady / 2))
        self.pad_left = int(np.ceil(padx / 2))
        self.pad_right = int(np.floor(padx / 2))

    def forward(self, x):
        return F.pad(x, [self.pad_left, self.pad_right, self.pad_top, self.pad_bottom], value=0)


class UnwrapPrepare(nn.Module):
    def forward(self, x):
        x = F.pad(x, [0, -1, 0, -1], value=0)
        y = x.contiguous().view(x.shape[0], -1)
        y = y.transpose(0, 1)
        return y.contiguous()

class UnwrapPool(nn.Module):
    def __init__(self, out_chans, cur_img_w, cur_img_h, d_w, d_h):
        super().__init__()
        self.out_chans = int(out_chans)
        self.cur_img_w = int(cur_img_w)
        self.cur_img_h = int(cur_img_h)
        self.d_w = int(d_w)
        self.d_h = int(d_h)

    def forward(self, x):
        y = x.view((self.out_chans, self.cur_img_w, self.cur_img_h, self.d_h, self.d_w, -1))
        y = y.transpose(2, 3)
        return y.contiguous()

class MultiMaxPooling(nn.Module):
    def __init__(self, k_w, k_h, d_w, d_h):
        super().__init__()
        self.paddings = []
        layers = []
        for i in range(d_h):
            for j in range(d_w):
                self.paddings.append((-j, -i))
                layers.append(nn.MaxPool2d(kernel_size=(k_w, k_h), stride=(d_w, d_h)))
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        heights, widths, outputs = [], [], []
        for layer, (pad_left, pad_top) in zip(self.layers, self.paddings):
            y = F.pad(x, [pad_left, pad_left, pad_top, pad_top], value=0)
            y = layer(y)
            heights.append(y.shape[2])
            widths.append(y.shape[3])
            outputs.append(y)

        max_h = int(np.max(heights))
        max_w = int(np.max(widths))

        padded_outputs = []
        for y in outputs:
            h, w = y.shape[2], y.shape[3]
            pad_top = int(np.floor((max_h - h) / 2))
            pad_bottom = int(np.ceil((max_h - h) / 2))
            pad_left = int(np.floor((max_w - w) / 2))
            pad_right = int(np.ceil((max_w - w) / 2))
            padded_outputs.append(F.pad(y, [pad_left, pad_right, pad_top, pad_bottom], value=0))

        return torch.cat(padded_outputs, dim=0)

In [ ]:
class USPretrainedResNet18Teacher(nn.Module):
    def __init__(self, patch_size=33, patch_batch_size=1024):
        super().__init__()
        self.patch_size = int(patch_size)
        self.patch_batch_size = int(patch_batch_size)
        self.output_dim = 512
        self.model = timm.create_model(
            "resnet18",
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        ).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.register_buffer(
            "mean",
            torch.tensor(US_IMAGENET_MEAN).view(1, 3, 1, 1),
            persistent=False,
        )
        self.register_buffer(
            "std",
            torch.tensor(US_IMAGENET_STD).view(1, 3, 1, 1),
            persistent=False,
        )

    @torch.no_grad()
    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x)

    @torch.no_grad()
    def forward_patch(self, x, decode=False, return_latent=False):
        if x.shape[-2:] != (self.patch_size, self.patch_size):
            raise ValueError(
                f"This teacher expects patch inputs of size {(self.patch_size, self.patch_size)}, got {tuple(x.shape[-2:])}"
            )

        latent = self.forward(x)
        if not decode:
            return latent

        if return_latent:
            return latent, latent
        return latent

    @torch.no_grad()
    def dense_latent(self, x):
        b, _, h, w = x.shape
        pad = self.patch_size // 2
        x = F.pad(x, [pad, pad, pad, pad], value=0)
        patches = F.unfold(x, kernel_size=self.patch_size, stride=1)
        patches = patches.transpose(1, 2).reshape(-1, 3, self.patch_size, self.patch_size)

        feats = []
        for start in range(0, patches.shape[0], self.patch_batch_size):
            stop = start + self.patch_batch_size
            feats.append(self.forward(patches[start:stop]))

        feats = torch.cat(feats, dim=0)
        return feats.view(b, h, w, self.output_dim)

class USPatchCNN(nn.Module):
    def __init__(self, patch_size=33, output_dim=128):
        super().__init__()
        self.patch_size = int(patch_size)
        self.output_dim = int(output_dim)
        self.latent_dim = self.output_dim
        self.act = nn.LeakyReLU(5e-3)
        self.max_pool = nn.MaxPool2d(2, 2)
        self.multi_pool_prepare = MultiPoolPrepare(self.patch_size, self.patch_size)
        self.multi_max_pooling = MultiMaxPooling(2, 2, 2, 2)
        self.unwrap_prepare = UnwrapPrepare()

        if self.patch_size == 17:
            self.conv1 = nn.Conv2d(3, 128, 6, 1)
            self.conv2 = nn.Conv2d(128, 256, 5, 1)
            self.conv3 = nn.Conv2d(256, 256, 5, 1)
            self.conv4 = nn.Conv2d(256, self.output_dim, 4, 1)
            self.out_chans = self.output_dim
            self.pool_stages = 0
        elif self.patch_size == 33:
            self.conv1 = nn.Conv2d(3, 128, 5, 1)
            self.conv2 = nn.Conv2d(128, 256, 5, 1)
            self.conv3 = nn.Conv2d(256, 256, 2, 1)
            self.conv4 = nn.Conv2d(256, self.output_dim, 4, 1)
            self.out_chans = self.output_dim
            self.pool_stages = 2
        elif self.patch_size == 65:
            self.conv1 = nn.Conv2d(3, 128, 5, 1)
            self.conv2 = nn.Conv2d(128, 128, 5, 1)
            self.conv3 = nn.Conv2d(128, 256, 5, 1)
            self.conv4 = nn.Conv2d(256, 256, 4, 1)
            self.conv5 = nn.Conv2d(256, self.output_dim, 1, 1)
            self.out_chans = self.output_dim
            self.pool_stages = 3
        else:
            raise ValueError("patch_size must be one of {17, 33, 65}")

        self.decoder = nn.Linear(self.latent_dim, 512)

    def forward_patch(self, x, decode=False, return_latent=False):
        if x.shape[-2:] != (self.patch_size, self.patch_size):
            raise ValueError(
                f"This network expects patch inputs of size {(self.patch_size, self.patch_size)}, got {tuple(x.shape[-2:])}"
            )

        if self.patch_size == 17:
            x = self.act(self.conv1(x))
            x = self.act(self.conv2(x))
            x = self.act(self.conv3(x))
            x = self.act(self.conv4(x))
        elif self.patch_size == 33:
            x = self.act(self.conv1(x))
            x = self.max_pool(x)
            x = self.act(self.conv2(x))
            x = self.max_pool(x)
            x = self.act(self.conv3(x))
            x = self.act(self.conv4(x))
        else:
            x = self.act(self.conv1(x))
            x = self.max_pool(x)
            x = self.act(self.conv2(x))
            x = self.max_pool(x)
            x = self.act(self.conv3(x))
            x = self.max_pool(x)
            x = self.act(self.conv4(x))
            x = self.act(self.conv5(x))

        latent = x.view(x.shape[0], -1)

        if not decode:
            return latent

        decoded = self.decoder(latent)
        return (latent, decoded) if return_latent else decoded

    def dense_latent(self, x):
        h, w = x.shape[-2:]
        divisor = 2 ** self.pool_stages
        if h % divisor != 0 or w % divisor != 0:
            raise ValueError(
                f"For patch_size={self.patch_size}, IMAGE_SIZE must be divisible by {divisor}; got {(h, w)}"
            )

        x = self.multi_pool_prepare(x)

        if self.patch_size == 17:
            x = self.act(self.conv1(x))
            x = self.act(self.conv2(x))
            x = self.act(self.conv3(x))
            x = self.act(self.conv4(x))
        elif self.patch_size == 33:
            unwrap_pool_2 = UnwrapPool(self.out_chans, h // 4, w // 4, 2, 2)
            unwrap_pool_1 = UnwrapPool(self.out_chans, h // 2, w // 2, 2, 2)
            x = self.act(self.conv1(x))
            x = self.multi_max_pooling(x)
            x = self.act(self.conv2(x))
            x = self.multi_max_pooling(x)
            x = self.act(self.conv3(x))
            x = self.act(self.conv4(x))
            x = self.unwrap_prepare(x)
            x = unwrap_pool_2(x)
            x = unwrap_pool_1(x)
        else:
            unwrap_pool_3 = UnwrapPool(self.out_chans, h // 8, w // 8, 2, 2)
            unwrap_pool_2 = UnwrapPool(self.out_chans, h // 4, w // 4, 2, 2)
            unwrap_pool_1 = UnwrapPool(self.out_chans, h // 2, w // 2, 2, 2)
            x = self.act(self.conv1(x))
            x = self.multi_max_pooling(x)
            x = self.act(self.conv2(x))
            x = self.multi_max_pooling(x)
            x = self.act(self.conv3(x))
            x = self.multi_max_pooling(x)
            x = self.act(self.conv4(x))
            x = self.act(self.conv5(x))
            x = self.unwrap_prepare(x)
            x = unwrap_pool_3(x)
            x = unwrap_pool_2(x)
            x = unwrap_pool_1(x)

        y = x.view(self.out_chans, h, w, -1)
        y = y.permute(3, 1, 2, 0).contiguous()
        return y


class UninformedStudentsLite:
    def __init__(
        self,
        patch_sizes=US_PATCH_SIZES,
        n_students=US_N_STUDENTS,
        teacher_epochs=US_TEACHER_EPOCHS,
        student_epochs=US_STUDENT_EPOCHS,
        teacher_lr=US_TEACHER_LR,
        student_lr=US_STUDENT_LR,
        weight_decay=US_WEIGHT_DECAY,
        teacher_image_batch_size=US_TEACHER_IMAGE_BATCH_SIZE,
        teacher_patch_batch_size=1024,
        student_batch_size=US_STUDENT_BATCH_SIZE,
        patches_per_image=US_PATCHES_PER_IMAGE,
        compactness_weight=US_COMPACTNESS_WEIGHT,
        sigma=US_ANOMALY_MAP_SIGMA,
    ):
        self.patch_sizes = tuple(int(p) for p in patch_sizes)
        self.n_students = int(n_students)
        self.teacher_epochs = int(teacher_epochs)
        self.student_epochs = int(student_epochs)
        self.teacher_lr = float(teacher_lr)
        self.student_lr = float(student_lr)
        self.weight_decay = float(weight_decay)
        self.teacher_image_batch_size = int(teacher_image_batch_size)
        self.teacher_patch_batch_size = int(teacher_patch_batch_size)
        self.student_batch_size = int(student_batch_size)
        self.patches_per_image = int(patches_per_image)
        self.compactness_weight = float(compactness_weight)
        self.sigma = float(sigma)
        self.scales = []

    def _train_teacher(self, teacher, distill_target, train_loader):
        teacher.eval()
        return teacher

    @torch.no_grad()
    def _compute_teacher_stats(self, teacher, train_loader):
        teacher.eval()
        sum_vec = None
        sum_sq_vec = None
        count = 0

        for batch in tqdm(train_loader, desc=f"Teacher stats p={teacher.patch_size}", leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            desc = teacher.dense_latent(images)
            sum_vec, sum_sq_vec, count = us_update_vector_stats(sum_vec, sum_sq_vec, count, desc)

        mean, var, std = us_finalize_vector_stats(sum_vec, sum_sq_vec, count)
        return {"mean": mean, "var": var, "std": std}

    def _train_students(self, students, teacher, teacher_stats, train_loader):
        teacher.eval()
        optimizers = [
            torch.optim.Adam(student.parameters(), lr=self.student_lr, weight_decay=self.weight_decay)
            for student in students
        ]
        best_states = [copy.deepcopy(student.state_dict()) for student in students]
        best_losses = [float("inf")] * len(students)

        mu = teacher_stats["mean"].view(1, 1, 1, -1)
        std = teacher_stats["std"].view(1, 1, 1, -1)

        for epoch in range(1, self.student_epochs + 1):
            epoch_losses = [[] for _ in students]
            for student in students:
                student.train()

            for batch in tqdm(train_loader, desc=f"Students p={teacher.patch_size} epoch {epoch}/{self.student_epochs}", leave=False):
                images = batch["image"].to(DEVICE, non_blocking=True)
                with torch.no_grad():
                    teacher_target = (teacher.dense_latent(images) - mu) / std

                for idx, (student, optimizer) in enumerate(zip(students, optimizers)):
                    optimizer.zero_grad(set_to_none=True)
                    student_pred = student.dense_latent(images)
                    loss = us_student_loss(student_pred, teacher_target)
                    loss.backward()
                    optimizer.step()
                    epoch_losses[idx].append(loss.item())

            mean_losses = [float(np.mean(losses)) for losses in epoch_losses]
            pretty = " | ".join([f"S{idx}: {loss:.6f}" for idx, loss in enumerate(mean_losses)])
            print(f"[US][students p={teacher.patch_size}] epoch {epoch:02d} | {pretty}")

            for idx, loss in enumerate(mean_losses):
                if loss < best_losses[idx]:
                    best_losses[idx] = loss
                    best_states[idx] = copy.deepcopy(students[idx].state_dict())

        for student, state in zip(students, best_states):
            student.load_state_dict(state)
            student.eval()

        return students

    @torch.no_grad()
    def _compute_score_stats(self, teacher, students, teacher_stats, val_loader):
        teacher.eval()
        for student in students:
            student.eval()

        mu = teacher_stats["mean"].view(1, 1, 1, -1)
        std = teacher_stats["std"].view(1, 1, 1, -1)

        err_sum = err_sum_sq = None
        err_count = 0
        var_sum = var_sum_sq = None
        var_count = 0

        for batch in tqdm(val_loader, desc=f"Score calibration p={teacher.patch_size}", leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            teacher_pred = (teacher.dense_latent(images) - mu) / std
            students_pred = torch.stack([student.dense_latent(images) for student in students], dim=1)

            err_map = us_regression_error_map(students_pred, teacher_pred)
            var_map = us_variance_map(students_pred)

            err_sum, err_sum_sq, err_count = us_update_scalar_stats(err_sum, err_sum_sq, err_count, err_map)
            var_sum, var_sum_sq, var_count = us_update_scalar_stats(var_sum, var_sum_sq, var_count, var_map)

        err_mean, err_var, err_std = us_finalize_scalar_stats(err_sum, err_sum_sq, err_count)
        var_mean, var_var, var_std = us_finalize_scalar_stats(var_sum, var_sum_sq, var_count)

        return {
            "err_mean": err_mean,
            "err_var": err_var,
            "err_std": err_std,
            "var_mean": var_mean,
            "var_var": var_var,
            "var_std": var_std,
        }

    @torch.no_grad()
    def _score_single_scale(self, images, scale):
        teacher = scale["teacher"]
        students = scale["students"]
        teacher_stats = scale["teacher_stats"]
        score_stats = scale["score_stats"]

        mu = teacher_stats["mean"].view(1, 1, 1, -1)
        std = teacher_stats["std"].view(1, 1, 1, -1)

        teacher_pred = (teacher.dense_latent(images) - mu) / std
        students_pred = torch.stack([student.dense_latent(images) for student in students], dim=1)

        err_map = us_regression_error_map(students_pred, teacher_pred)
        var_map = us_variance_map(students_pred)

        score_map = (err_map - score_stats["err_mean"]) / score_stats["err_std"]
        score_map = score_map + (var_map - score_stats["var_mean"]) / score_stats["var_std"]
        return score_map

    def fit(self, train_dataset, val_dataset):
        train_dense_loader = us_make_loader(
            train_dataset,
            batch_size=self.student_batch_size,
            shuffle=True,
        )
        train_dense_stats_loader = us_make_loader(
            train_dataset,
            batch_size=self.student_batch_size,
            shuffle=False,
        )
        val_dense_loader = us_make_loader(
            val_dataset,
            batch_size=self.student_batch_size,
            shuffle=False,
        )

        self.scales = []

        for patch_size in self.patch_sizes:
            teacher = USPretrainedResNet18Teacher(
                patch_size=patch_size,
                patch_batch_size=self.teacher_patch_batch_size,
            ).to(DEVICE).eval()
            students = [USPatchCNN(patch_size, output_dim=teacher.output_dim).to(DEVICE) for _ in range(self.n_students)]
            print(
                f"[US] patch size {patch_size} | teacher params={us_count_parameters(teacher):,} "
                f"(frozen pretrained ResNet18) | students={len(students)}"
            )

            teacher_stats = self._compute_teacher_stats(teacher, train_dense_stats_loader)
            students = self._train_students(students, teacher, teacher_stats, train_dense_loader)
            score_stats = self._compute_score_stats(teacher, students, teacher_stats, val_dense_loader)

            self.scales.append(
                {
                    "patch_size": patch_size,
                    "teacher": teacher,
                    "students": students,
                    "teacher_stats": teacher_stats,
                    "score_stats": score_stats,
                }
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return self

    @torch.no_grad()
    def predict(self, loader):
        if len(self.scales) == 0:
            raise RuntimeError("Call fit(...) before predict(...)")

        results = []
        for batch in tqdm(loader, desc="Uninformed Students inference"):
            images = batch["image"].to(DEVICE, non_blocking=True)

            score_maps = [self._score_single_scale(images, scale) for scale in self.scales]
            score_map = torch.stack(score_maps, dim=0).mean(dim=0)
            score_map = score_map.unsqueeze(1)

            if self.sigma is not None and self.sigma > 0:
                score_map = gaussian_blur(score_map, sigma=self.sigma)

            image_scores = score_map.flatten(1).amax(dim=1)
            score_map = score_map.squeeze(1)

            for i in range(images.shape[0]):
                results.append(
                    {
                        "image": tensor_to_numpy_image(images[i]),
                        "anomaly_map": score_map[i].detach().cpu().numpy().astype(np.float32),
                        "mask": tensor_to_numpy_mask(batch["mask"][i]),
                        "label": int(batch["label"][i].item()),
                        "score": float(image_scores[i].item()),
                        "defect_type": batch["defect_type"][i],
                        "path": batch["path"][i],
                    }
                )

        return results

In [ ]:
torch.cuda.empty_cache()

uninformed_students_test_loader = us_make_loader(
    test_dataset,
    batch_size=1,
    shuffle=False,
)

uninformed_students = UninformedStudentsLite(
    patch_sizes=US_PATCH_SIZES,
    n_students=US_N_STUDENTS,
    teacher_epochs=US_TEACHER_EPOCHS,
    student_epochs=US_STUDENT_EPOCHS,
    teacher_lr=US_TEACHER_LR,
    student_lr=US_STUDENT_LR,
    weight_decay=US_WEIGHT_DECAY,
    teacher_image_batch_size=US_TEACHER_IMAGE_BATCH_SIZE,
    student_batch_size=US_STUDENT_BATCH_SIZE,
    patches_per_image=US_PATCHES_PER_IMAGE,
    compactness_weight=US_COMPACTNESS_WEIGHT,
    sigma=US_ANOMALY_MAP_SIGMA,
)


In [ ]:
# Train the model
#uninformed_students.fit(train_dataset, val_dataset)
#torch.save(uninformed_students, "/content/drive/MyDrive/uninformed_students_10epochs_resnet18.pth")

# Comment if you want to train from scratch
uninformed_students = torch.load(
    "/content/drive/MyDrive/uninformed_students_10epochs_resnet18.pth", weights_only=False, map_location=torch.device(DEVICE)
)
uninformed_students_test_results = uninformed_students.predict(uninformed_students_test_loader)
uninformed_students_metrics = summarize_metrics(uninformed_students_test_results)
uninformed_students_thresholds = {
    k: uninformed_students_metrics[k] for k in ("pixel_threshold", "image_threshold")
}

us_scale_name = ",".join(str(p) for p in uninformed_students.patch_sizes)
print(f"Uninformed Students Lite metrics (p={us_scale_name}, M={uninformed_students.n_students}):")
display(pd.DataFrame([uninformed_students_metrics]).round(4))

plot_score_distribution_v2(
    uninformed_students_test_results,
    title=f"Uninformed Students Lite image scores (p={us_scale_name})",
)

show_detection_results(
    uninformed_students_test_results,
    n=N_VIS_SAMPLES,
    include_reconstruction=False,
    thresholds=uninformed_students_thresholds,
    title=f"Uninformed Students Lite: test examples (p={us_scale_name})",
)


## Part 3 — PatchCore as a feature-based anomaly detector

PatchCore changes the perspective completely. Instead of reconstructing the image, we extract **patch features** from a pretrained CNN and keep a **memory bank of normal patches**. At test time, each test patch is compared to the memory bank, if a test patch is **far from every normal patch**, it is considered anomalous.

### Main ingredients

1. **Pretrained backbone**  
   We use a pretrained CNN (preferably **WideResNet50**, with an automatic fallback to a lighter model if needed).

2. **Intermediate feature maps**  
   We use mid-level features (roughly `layer2` and `layer3`) because they contain both texture and semantic information.

3. **Local aggregation**  
   We average features in a small local neighborhood (`patchsize = 3`) to make patch descriptors more context-aware.

4. **Memory bank + coreset**  
   Keeping every patch is often too heavy.  
   We therefore keep a smaller but diverse subset of normal patches with a **greedy coreset** strategy.

5. **Nearest-neighbor distance**
   For each test patch, we compute its distance to the closest normal patch in the memory bank.

6. **Explainability**  
   For a highly anomalous test patch, we can retrieve the **nearest normal patch** that the model considered "closest".  
   This helps explain *why* the patch is unusual.


In [ ]:
# -------------------------
# PatchCore Lite settings
# -------------------------

# We try WideResNet50 first, then automatically fall back to a lighter model
# if the runtime cannot handle it.
PATCHCORE_BACKBONE_CANDIDATES = ["wide_resnet50_2", "resnet18"]

# In timm's ResNet feature extractor, indices (2, 3) correspond roughly to
# ResNet layer2 and layer3, which is close to the PatchCore setup.
PATCHCORE_OUT_INDICES = (2, 3)

# Local aggregation window size for patch features.
PATCHCORE_PATCHSIZE = 3

# Coreset parameters.
# Smaller values are faster and lighter; larger values can improve performance.
PATCHCORE_CORESET_FRACTION = 0.01
PATCHCORE_CANDIDATE_POOL_SIZE = 6000
PATCHCORE_RANDOM_PROJECTION_DIM = 64

# Exact nearest-neighbor search is done in chunks (no FAISS).
PATCHCORE_BANK_CHUNK_SIZE = 2048

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# This class helps us extract the features from a pre-trained backbone.
# It also performs a sanity check on the runtime memory availability

class TimmFeatureExtractor(nn.Module):
    def __init__(self, backbone_candidates, out_indices=(2, 3)):
        super().__init__()
        self.out_indices = tuple(out_indices)
        self.backbone_name, self.model = self._build_model(backbone_candidates, self.out_indices)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)

    def _build_model(self, backbone_candidates, out_indices):
        last_err = None
        for backbone_name in backbone_candidates:

            # We do a check on runtime memory to see if the model fits
            try:
                model = timm.create_model(
                    backbone_name,
                    pretrained=True,
                    features_only=True,
                    out_indices=out_indices,
                ).to(DEVICE).eval()

                with torch.no_grad():
                    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
                    mean = dummy.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
                    std = dummy.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
                    _ = model((dummy - mean) / std)

                print(f"Using PatchCore backbone: {backbone_name}")
                return backbone_name, model
            except Exception as err:
                print(f"Backbone '{backbone_name}' failed on this runtime: {err}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                last_err = err

        raise RuntimeError(f"All candidate backbones failed. Last error: {last_err}")

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x)

# Helper function to randomly project the features in a lower dimensional space
def random_project(x, out_dim=64, seed=SEED):
    if out_dim is None or x.shape[1] <= out_dim:
        return x
    generator = torch.Generator(device=x.device)
    generator.manual_seed(seed)
    proj = torch.randn(x.shape[1], out_dim, generator=generator, device=x.device, dtype=x.dtype)
    proj = F.normalize(proj, dim=0)
    return x @ proj

# Greedy coreset procedure to subsample the memory bank
def greedy_coreset(points, k, projection_dim=64, seed=SEED):
    ...

# This function performs a knn chunk-wise
def knn_search(query, bank, bank_chunk_size=2048):
    query = query.float()
    bank = bank.float()

    best_dist = torch.full((query.shape[0],), float("inf"), dtype=torch.float32)
    best_idx = torch.full((query.shape[0],), -1, dtype=torch.long)

    for start in range(0, bank.shape[0], bank_chunk_size):
        bank_chunk = bank[start:start + bank_chunk_size]
        dists = torch.cdist(query, bank_chunk)
        vals, inds = dists.min(dim=1)
        update = vals < best_dist
        best_dist[update] = vals[update]
        best_idx[update] = inds[update] + start

    return best_dist, best_idx

class PatchCoreLite:
    def __init__(
        self,
        backbone_candidates=("wide_resnet50_2", "resnet18"),
        out_indices=(2, 3),
        patchsize=3,
        coreset_fraction=0.01,
        candidate_pool_size=6000,
        projection_dim=64,
        bank_chunk_size=2048,
        sigma=4.0,
    ):
        self.feature_extractor = TimmFeatureExtractor(backbone_candidates, out_indices).to(DEVICE).eval()
        self.backbone_name = self.feature_extractor.backbone_name
        self.out_indices = tuple(out_indices)
        self.patchsize = int(patchsize)
        self.coreset_fraction = float(coreset_fraction)
        self.candidate_pool_size = int(candidate_pool_size)
        self.projection_dim = projection_dim
        self.bank_chunk_size = int(bank_chunk_size)
        self.sigma = float(sigma)

        self.memory_bank = None
        self.bank_image_ids = None
        self.bank_coords = None
        self.feature_grid_shape = None
        self.train_images = []
        self.train_paths = []

        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            feat_shapes = [tuple(f.shape) for f in self.feature_extractor(dummy)]
        print("Selected feature map shapes:", feat_shapes)

    @torch.no_grad()
    def _extract_patch_embeddings(self, images):
        ...

    @torch.no_grad()
    def fit(self, loader):
        self.feature_extractor.eval()
        self.train_images = []
        self.train_paths = []

        all_embeddings = []
        all_image_ids = []
        all_coords = []
        image_counter = 0
        coords_cache = {}

        for batch in tqdm(loader, desc="PatchCore feature extraction"):
            ...

            if (h, w) not in coords_cache:
                yy, xx = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
                coords_cache[(h, w)] = torch.stack([yy, xx], dim=-1).reshape(-1, 2).long()

            base_coords = coords_cache[(h, w)]

            for i in range(b):
                self.train_images.append((tensor_to_numpy_image(images[i]) * 255).astype(np.uint8))
                self.train_paths.append(batch["path"][i])
                all_image_ids.append(torch.full((h * w,), image_counter, dtype=torch.long))
                all_coords.append(base_coords.clone())
                image_counter += 1

        full_bank = torch.cat(all_embeddings, dim=0)
        full_image_ids = torch.cat(all_image_ids, dim=0)
        full_coords = torch.cat(all_coords, dim=0)

        print(f"Full normal patch bank: {full_bank.shape[0]:,} patches x {full_bank.shape[1]} dims")

        n_total = full_bank.shape[0]
        n_coreset = max(1, int(round(n_total * self.coreset_fraction)))

        # This is a just a safety check not to have too many features
        n_candidate = min(n_total, max(n_coreset, self.candidate_pool_size))

        rng = torch.randperm(n_total)
        candidate_idx = rng[:n_candidate]
        candidate_bank = full_bank[candidate_idx].cpu()

        selected_in_candidate = greedy_coreset(
            candidate_bank,
            k=n_coreset,
            projection_dim=self.projection_dim,
            seed=SEED,
        )
        selected_idx = candidate_idx[selected_in_candidate]

        self.memory_bank = full_bank[selected_idx].contiguous().cpu()
        self.bank_image_ids = full_image_ids[selected_idx].contiguous().cpu()
        self.bank_coords = full_coords[selected_idx].contiguous().cpu()

        print(f"Coreset memory bank: {self.memory_bank.shape[0]:,} patches")
        return self

    @torch.no_grad()
    def predict(self, loader):
        if self.memory_bank is None:
            raise RuntimeError("PatchCoreLite has not been fitted yet.")

        results = []
        bank = self.memory_bank.cpu()

        for batch in tqdm(loader, desc="PatchCore inference"):
            ...

            for i in range(b):
                results.append(
                    {
                        "image": tensor_to_numpy_image(images[i]),
                        "anomaly_map": anomaly_maps[i, 0].detach().cpu().numpy().astype(np.float32),
                        "mask": tensor_to_numpy_mask(batch["mask"][i]),
                        "label": int(batch["label"][i].item()),
                        "score": float(image_scores[i].item()),
                        "defect_type": batch["defect_type"][i],
                        "path": batch["path"][i],
                        "patch_scores_small": patch_scores_small[i].detach().cpu().numpy().astype(np.float32),
                        "nn_indices_small": nn_indices_small[i].detach().cpu().numpy().astype(np.int64),
                    }
                )

        return results


Helper functions to draw the nearest patch in the visualization results

In [ ]:
def draw_box(ax, bbox, color="lime", linewidth=2):
    x0, y0, x1, y1 = bbox
    rect = plt.Rectangle(
        (x0, y0),
        max(1, x1 - x0),
        max(1, y1 - y0),
        fill=False,
        edgecolor=color,
        linewidth=linewidth,
    )
    ax.add_patch(rect)

def grid_coord_to_bbox(coord, image_shape, grid_shape):
    gy, gx = int(coord[0]), int(coord[1])
    h_img, w_img = image_shape[:2]
    h_grid, w_grid = grid_shape

    y0 = int(round(gy * h_img / h_grid))
    y1 = int(round((gy + 1) * h_img / h_grid))
    x0 = int(round(gx * w_img / w_grid))
    x1 = int(round((gx + 1) * w_img / w_grid))

    y0 = max(0, min(y0, h_img - 1))
    x0 = max(0, min(x0, w_img - 1))
    y1 = max(y0 + 1, min(y1, h_img))
    x1 = max(x0 + 1, min(x1, w_img))
    return x0, y0, x1, y1

def crop_with_bbox(image, bbox):
    x0, y0, x1, y1 = bbox
    return image[y0:y1, x0:x1]

def explain_patchcore_match(patchcore, results, rank=0):
    if len(results) == 0:
        print("No results available.")
        return

    order = np.argsort([r["score"] for r in results])[::-1]
    rank = int(np.clip(rank, 0, len(order) - 1))
    r = results[order[rank]]

    patch_scores = r["patch_scores_small"]
    y, x = np.unravel_index(np.argmax(patch_scores), patch_scores.shape)

    bank_idx = int(r["nn_indices_small"][y, x])
    source_img_id = int(patchcore.bank_image_ids[bank_idx].item())
    source_coord = patchcore.bank_coords[bank_idx].tolist()

    test_img = r["image"]
    train_img = patchcore.train_images[source_img_id].astype(np.float32) / 255.0

    test_bbox = grid_coord_to_bbox((y, x), test_img.shape, patch_scores.shape)
    train_bbox = grid_coord_to_bbox(source_coord, train_img.shape, patchcore.feature_grid_shape)

    test_patch = crop_with_bbox(test_img, test_bbox)
    train_patch = crop_with_bbox(train_img, train_bbox)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

    axes[0].imshow(test_img)
    axes[0].imshow(normalize_map(r["anomaly_map"]), cmap="jet", alpha=0.35)
    draw_box(axes[0], test_bbox)
    axes[0].set_title(f"Query test image\nscore={r['score']:.4f}")
    axes[0].axis("off")

    axes[1].imshow(test_patch)
    axes[1].set_title("Most anomalous query patch")
    axes[1].axis("off")

    axes[2].imshow(train_img)
    draw_box(axes[2], train_bbox)
    axes[2].set_title("Nearest normal training image")
    axes[2].axis("off")

    axes[3].imshow(train_patch)
    axes[3].set_title("Nearest normal patch")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    print("Query image:", r["path"])
    print("Nearest normal image:", patchcore.train_paths[source_img_id])
    print("Query patch grid coordinate:", (int(y), int(x)))
    print("Nearest normal patch grid coordinate:", tuple(int(v) for v in source_coord))

We fill the memory bank and perform the evaluation

In [ ]:
torch.cuda.empty_cache()

patchcore = PatchCoreLite(
    backbone_candidates=PATCHCORE_BACKBONE_CANDIDATES,
    out_indices=PATCHCORE_OUT_INDICES,
    patchsize=PATCHCORE_PATCHSIZE,
    coreset_fraction=0.01,
    candidate_pool_size=PATCHCORE_CANDIDATE_POOL_SIZE,
    projection_dim=512,
    bank_chunk_size=PATCHCORE_BANK_CHUNK_SIZE,
    sigma=ANOMALY_MAP_SIGMA,
)

patchcore.fit(patchcore_train_loader)

patchcore_val_results = patchcore.predict(val_loader)

patchcore_test_results = patchcore.predict(test_loader)
patchcore_metrics = summarize_metrics(patchcore_test_results)
patchcore_thresholds = {k: patchcore_metrics[k] for k in ("pixel_threshold", "image_threshold")}

print(f"PatchCore Lite metrics (backbone: {patchcore.backbone_name}):")
display(pd.DataFrame([patchcore_metrics]).round(4))

plot_score_distribution_v2(
    patchcore_test_results,
    title=f"PatchCore Lite image scores ({patchcore.backbone_name})",
)

show_detection_results(
    patchcore_test_results,
    n=N_VIS_SAMPLES,
    include_reconstruction=False,
    thresholds=patchcore_thresholds,
    title=f"PatchCore Lite: hardest test examples ({patchcore.backbone_name})",
)

# Visual explanation for the highest-scoring test image
explain_patchcore_match(patchcore, patchcore_test_results, rank=0)

Now we compare the performance of all the proposed algorithms

In [ ]:
comparison_rows = [
    {"model": "Convolutional Latent Space Autoencoder (Improved Reconstruction Loss)", **conv_autoencoder_mse_metrics},
    {"model": "Bottleneck Autoencoder (MSE)", **bott_autoencoder_mse_metrics},
    {
        "model": f"Uninformed Students Lite (p={','.join(map(str, uninformed_students.patch_sizes))}, M={uninformed_students.n_students})",
        **uninformed_students_metrics,
    },
    {"model": f"PatchCore Lite ({patchcore.backbone_name})", **patchcore_metrics},
]

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

comparison_df.set_index("model")[["image_auroc", "image_ap", "pixel_auroc", "pixel_ap"]].plot(
    kind="bar",
    figsize=(11, 4),
    rot=20,
)
plt.title(f"Model comparison on MVTec AD / {CLASS_NAME}")
plt.ylabel("score")
plt.ylim(0.0, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.show()

## Part 4 Deep Anomaly Detection Using Geometric Transformations

Here we imlpement the paper "Deep Anomaly Detection Using Geometric Transformations". The paper lavarage a big Wide Residual Network (WRN), trained for 50 epochs.

Then, the paper define 72 transformations that are applied to the image. As from the paper the transformations are from 3 groups:
- Horizontal flip
- Translation
- Rotation of multiple of 90 degrees

Combining all possible values of these (2x3x3x4) we get 72 possible values

In particular our code fist define the combination of the transformations using four nested loops (two are required for the shig). We proceed from flip (true/false), vertical shift (up/none/down), horizontal shifft (left/none/right), and rotations (0/90/180/270). This gives us a vector which defines our transformation.

Then, given an image and a vector which defines our transformation we apply them to the input image and return the final transformed image.

In [ ]:
class GeometricTransformations:
    def __init__(self):
        self.transforms = []
        for flip in [False, True]:  # Horizontal flip
            for sh in [-1, 0, 1]:  # Vertical shift
                for sw in [-1, 0, 1]:  # Horizontal shift
                    for rot in [0, 90, 180, 270]:  # Rotation
                        self.transforms.append((flip, sh, sw, rot))

    def apply(self, img, transform_index):
        ...

    def get_num_transforms(self):
        return len(self.transforms)

Next, we have to define our network, a Wide Residual Network. We have to first manually define the residual block as a series of convolutions, Batch normalization and relus

In [ ]:
# Define a Residual Block
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        ...

    def forward(self, x):
        ...
        return out


Now that we have defined the Residual Block we can proceed with the network definition. The network start with a convolution, followed by three residual blocks, and a final layer to map the number of classes.

The depth parameter regulate the number of residual block that are added by the _make_layer function.

The widen_factor instead regulates the number of channel in the network

In [ ]:
# Define the Wide Residual Network (WRN)
class WRN(nn.Module):
    def __init__(self, depth=16, widen_factor=8, num_classes=72):
        super(WRN, self).__init__()
        assert (depth - 4) % 6 == 0, "Depth should be 6n+4"
        n = (depth - 4) // 6
        channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]

        self.conv1 = nn.Conv2d(3, channels[0], kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels[0])
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(channels[0], channels[1], n, stride=1)
        self.layer2 = self._make_layer(channels[1], channels[2], n, stride=2)
        self.layer3 = self._make_layer(channels[2], channels[3], n, stride=2)

        self.bn2 = nn.BatchNorm2d(channels[3])
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels[3], num_classes)

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.avg_pool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out

We now define an helper class to create our custom dataset, we take as input the original dataset, with his classes, like cat or dog, and given an image it ammpy a random transformation from the 72 possible transformations

In [ ]:
class MVTecTransformedSingleClassDataset(Dataset):
    def __init__(self, root, class_name, split="train", image_size=256, transformations=None):
        self.root = Path(root)
        self.class_name = class_name
        self.split = split
        self.image_size = image_size
        self.samples = []
        self.class_root = self.root / class_name
        self.transformations = transformations

        if split not in {"train", "test"}:
            raise ValueError("split must be 'train' or 'test'")

        if split == "train":
            good_dir = self.class_root / "train" / "good"
            for img_path in list_image_files(good_dir):
                self.samples.append(
                    {
                        "image_path": img_path,
                        "mask_path": None,
                        "label": 0,
                        "defect_type": "good",
                    }
                )
        else:
            test_root = self.class_root / "test"
            gt_root = self.class_root / "ground_truth"
            for defect_dir in sorted(test_root.iterdir()):
                if not defect_dir.is_dir():
                    continue
                defect_type = defect_dir.name
                for img_path in list_image_files(defect_dir):
                    if defect_type == "good":
                        mask_path = None
                        label = 0
                    else:
                        mask_name = f"{img_path.stem}_mask.png"
                        mask_path = gt_root / defect_type / mask_name
                        label = 1
                    self.samples.append(
                        {
                            "image_path": img_path,
                            "mask_path": mask_path,
                            "label": label,
                            "defect_type": defect_type,
                        }
                    )

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No samples found for class='{class_name}', split='{split}' under {self.class_root}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        record = self.samples[idx]

        image = Image.open(record["image_path"]).convert("RGB")
        image = image.resize((self.image_size, self.image_size), resample=Image.BILINEAR)

        transform_idx = np.random.randint(0, self.transformations.get_num_transforms())
        transformed_img = self.transformations.apply(image, transform_idx)

        image = pil_to_tensor(transformed_img)

        if record["mask_path"] is None:
            mask = torch.zeros((1, self.image_size, self.image_size), dtype=torch.float32)
        else:
            mask = Image.open(record["mask_path"]).convert("L")
            mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)
            mask = (pil_to_tensor(mask) > 0.5).float()

        return {
            "image": image,
            "mask": mask,
            "label": torch.tensor(record["label"], dtype=torch.long),
            "defect_type": record["defect_type"],
            "path": str(record["image_path"]),
            "transformed_idx": torch.tensor(transform_idx, dtype=torch.long)
        }

def split_train_val(dataset, val_ratio=0.15, seed=SEED):
    n = len(dataset)
    indices = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    n_val = max(1, int(round(n * val_ratio)))
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]
    return Subset(dataset, train_idx.tolist()), Subset(dataset, val_idx.tolist())


In [ ]:
transformations = GeometricTransformations()
train_full_dataset = MVTecTransformedSingleClassDataset(MVTEC_ROOT, CLASS_NAME, split="train", image_size=IMAGE_SIZE, transformations=transformations)
test_dataset = MVTecSingleClassDataset(MVTEC_ROOT, CLASS_NAME, split="test", image_size=IMAGE_SIZE)
train_dataset, val_dataset = split_train_val(train_full_dataset, val_ratio=VAL_RATIO, seed=SEED)

print(f"Train normals (full): {len(train_full_dataset)}")
print(f"Train normals (used for fitting): {len(train_dataset)}")
print(f"Validation normals: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=AE_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=AE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=PATCHCORE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)


Now we have the main train function

We set some variables for early stopping

Then we iterate thorugh epochs and images

We monitor the loss each epoch and save models with lower loss

We follow the original work to define a way to discriminate between normal and anomaly images. Given an image, computes all the transfomration of the image and predict the probability of each one of these image. The score is computed as the mean of the probability of the correct class. So, given the input this is the diagonal of the softnamx scores.

This is the simplified function shown in the paper. An alternative solution is to use the Dirichlet Normality Score

In [ ]:
def plot_history(history_df, title="Training history"):
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.show()

def train_self_supervised(model, train_loader, epochs, optimizer, criterion):

    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for batch in tqdm(train_loader, desc=f"Self-Supervised Net train epoch {epoch}/{epochs}", leave=False):
            ...

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                ...

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"[Self-Supervised Net] epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df

@torch.no_grad()
def predict_self_supervised(model, loader, transformations):
    model.eval()
    results = []

    for batch in tqdm(loader, desc=f"Self-Supervised inference"):
        x = batch["image"].to(DEVICE, non_blocking=True)

        ...
    return results


In [ ]:
# Initialize model, loss, optimizer
ss_model = WRN(depth=16, widen_factor=8, num_classes=72).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(ss_model.parameters(), lr=0.0001)
SS_EPOCHS = 50

And we start the training

In [ ]:
# Uncomment the following lines if you want to train
#self_sup_model, self_sup_history = train_self_supervised(ss_model, train_loader, SS_EPOCHS, optimizer, criterion)
#plot_history(self_sup_history, title="Self Supervised Model training history")
#torch.save(self_sup_model, "/content/drive/MyDrive/self_sup_lr_v2.pth")
#with open("/content/drive/MyDrive/self_sup_lr_v2_history.pkl", "wb") as f:
#    pickle.dump(self_sup_history, f)

# Comment if you want to train from scratch
self_sup_model = torch.load(
    "/content/drive/MyDrive/self_sup_lr_v2.pth", weights_only=False, map_location=torch.device(DEVICE)
).to(DEVICE)
with open("/content/drive/MyDrive/self_sup_lr_v2_history.pkl", "rb") as f:
    self_sup_history = pickle.load(f)
plot_history(self_sup_history, title="Self Supervised Model training history")


Now that we have trained our model we can proceed with the validation

In [ ]:
# Evaluate the model
self_sup_model_test_results = predict_self_supervised(self_sup_model, test_loader, transformations)
self_sup_metrics = summarize_metrics_image_only(self_sup_model_test_results)
print("Self Supervised Network metrics:")
display(pd.DataFrame([self_sup_metrics]).round(4))

plot_score_distribution_v2(self_sup_model_test_results, title="Self Supervised Model image scores")

In [ ]:
class MVTecTwoClassesDataset(Dataset):
    def __init__(self, root, class_name_1, class_name_2, image_size=256, subsampling_ratio=0.5):
        self.root = Path(root)
        self.class_name_1 = class_name_1
        self.class_name_2 = class_name_2
        self.image_size = image_size
        self.samples = []
        self.class_root_1 = self.root / class_name_1
        self.class_root_2 = self.root / class_name_2
        self.subsampling_ratio = subsampling_ratio

        all_samples = []

        good_dir = self.class_root_1 / "train" / "good"
        for img_path in list_image_files(good_dir):
            all_samples.append(
                {
                    "image_path": img_path,
                    "mask_path": None,
                    "label": 1,
                    "defect_type": "good",
                }
            )

        anomalous_dir = self.class_root_2 / "train" / "good"
        for img_path in list_image_files(anomalous_dir):
            all_samples.append(
                {
                    "image_path": img_path,
                    "mask_path": None,
                    "label": 0,
                    "defect_type": "other_class",
                }
            )

        # We remove some items from the sample list to speed up inference
        random.shuffle(all_samples)       # shuffle in place
        n = math.ceil(len(all_samples) * self.subsampling_ratio)

        self.samples = all_samples[:n]

        if len(self.samples) == 0:
            raise RuntimeError(
                f"No samples found for class='{class_name}', split='{split}' under {self.class_root_1} or {self.class_root_2}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        record = self.samples[idx]

        image = Image.open(record["image_path"]).convert("RGB")
        image = image.resize((self.image_size, self.image_size), resample=Image.BILINEAR)
        image = pil_to_tensor(image)

        return {
            "image": image,
            "label": torch.tensor(record["label"], dtype=torch.long),
            "mask": 0,
            "defect_type": record["defect_type"],
            "path": str(record["image_path"]),
        }

In [ ]:
test_dataset_two_classes = MVTecTwoClassesDataset(MVTEC_ROOT, CLASS_NAME, "cable", image_size=IMAGE_SIZE)
test_loader_two_classes = DataLoader(
    test_dataset_two_classes,
    batch_size=PATCHCORE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

In [ ]:
# Evaluate the model in a one-class classification scenario

self_sup_model_test_results_2 = predict_self_supervised(self_sup_model, test_loader_two_classes, transformations)
self_sup_metrics_2 = summarize_metrics_image_only(self_sup_model_test_results_2)
print("Self Supervised Network metrics:")
display(pd.DataFrame([self_sup_metrics_2]).round(4))

plot_score_distribution_v2(self_sup_model_test_results_2, title="MSE autoencoder image scores")


## Discussion

### 1) Autoencoders and Uninformed Students vs PatchCore

**Autoencoders** are intuitive and easy to explain:

- they learn a compressed representation of normal images
- anomalies appear where reconstruction is poor

But they also have weaknesses:

- they may reconstruct some anomalies surprisingly well
- pixel-space losses can be sensitive to alignment and low-level appearance

**PatchCore**, in contrast, does not reconstruct the image at all:

- it relies on strong pretrained features
- it compares every test patch to a bank of normal patches
- unusual feature patterns get large nearest-neighbor distances

This usually makes PatchCore especially strong on industrial datasets with rich textures and subtle defects.

---

## Suggested exercises

1. Change `CLASS_NAME` and compare algorithms performances on different categories such as `wood`, `capsule`, and `screw`.
2. Increase `IMAGE_SIZE` to `320` or `384` and observe memory/runtime trade-offs.
3. Explore autoencoder enhancement such as:
   - skip connections
   - a denoising objective
   - a more powerful architecture
4. Try to train a multiscale model for Uninformed Students
5. Replace the image score in PatchCore with a **top-k mean** instead of a pure max.
6. Try to increase/decrease the PatchCore sampling ratio
7. Try to increase the epochs for the Self-Supervised Method

---
